# Finetune 4 models on VisDrone — v8n / v11n / v26n / v26n-p2Chạy trên Colab A100. Notebook này train **4 model**, val theo đúng giao thứcVisDrone, export ONNX sẵn sàng cho pipeline QCS8550, và ghi lại manifest đểmọi con số sau này truy ngược được.---## Ba điều đã kiểm chứng trước khi viết notebook nàyCả ba đều làm thay đổi thiết kế, nên đọc trước khi chạy.### 1. Colab tiêu chuẩn chỉ cấp **1 GPU**, không phải 2Colab (kể cả Pro+) cấp một GPU mỗi runtime. Không có runtime 2 GPU.(Colab **Enterprise**/Vertex AI thì có, nhưng đó là sản phẩm khác.)Notebook **tự thích ứng**:| Số GPU thấy được | Cách chạy ||---|---|| ≥ 2 | mỗi model một GPU, thật sự song song || 1 (thường gặp) | 2 tiến trình cùng chạy trên một A100 |Chạy 2 tiến trình trên một A100 **vẫn nhanh hơn chạy lần lượt**: model nano ởbatch ~96 không lấp đầy được A100, nên hai job xen kẽ nhau lấp vào chỗ trống.Đừng kỳ vọng 2× — thực tế khoảng 1.5–1.7×.### 2. YOLO26 **không dùng NMS** — output khác hoàn toànĐo trực tiếp trên ultralytics 8.4.118:| Model | Output thô | Cần NMS? ||---|---|---|| `yolov8n` | `(1, 4+nc, 8400)` | ✅ có || `yolo11n` | `(1, 4+nc, 8400)` | ✅ có || **`yolo26n`** | **`(1, 300, 6)`** | ❌ **không** || **`yolo26n-p2`** | **`(1, 300, 6)`** | ❌ **không** |`(1, 300, 6)` là box đã decode sẵn `[x1, y1, x2, y2, conf, cls]`.**Hệ quả tốt:** pipeline hiện tại tốn **18 ms/frame** cho NMS trên CPU, so với47 ms inference. Bỏ được NMS là cắt gần 30% thời gian xử lý một frame.**Hệ quả cần xử lý:** `3-pipeline/detector.py` đang decode `(1, 4+nc, A)`. Nạpmodel v26 vào đó sẽ **sai thầm lặng**, không báo lỗi. Cell export ở cuối ghi rõshape của từng model vào manifest để không ai cắm nhầm.### 3. `yolo26n-p2` **không có pretrained** — chỉ 50% trọng số nạp đượcKhông tồn tại `yolo26n-p2.pt`. Phải dựng từ yaml rồi nạp một phần từ`yolo26n.pt`. Đo thử trên kiến trúc p2 tương đương: **219/437 tensor** đượcchuyển, tức **một nửa model khởi tạo ngẫu nhiên**.Nên v26n-p2 **xuất phát ở vạch khác ba model kia**. Nếu để cùng số epoch rồikết luận "p2 kém hơn" thì đó là kết luận sai — nó chỉ chưa hội tụ. Notebook mặcđịnh cho p2 **nhiều epoch hơn**, và ghi tỉ lệ transfer vào manifest để bảng sosánh nói rõ điều này.---## Thứ tự chạyCell 1 → 15 theo đúng thứ tự. Cell 11 và 12 là hai lượt train, mỗi lượt vàigiờ. Notebook có resume nếu Colab ngắt giữa chừng.

## 1. Kiểm tra GPUDừng ngay nếu không phải GPU mong đợi, thay vì phát hiện sau 3 tiếng.

In [ ]:
import subprocess, sysprint(subprocess.run(    ["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version",     "--format=csv,noheader"],    capture_output=True, text=True).stdout or "!! khong thay nvidia-smi")import torchN_GPU = torch.cuda.device_count()if N_GPU == 0:    raise SystemExit("Khong co GPU. Runtime > Change runtime type > GPU (A100).")names = [torch.cuda.get_device_name(i) for i in range(N_GPU)]vram = [torch.cuda.get_device_properties(i).total_memory / 1e9 for i in range(N_GPU)]for i, (n, v) in enumerate(zip(names, vram)):    print(f"  GPU {i}: {n}  {v:.1f} GB")IS_A100 = any("A100" in n for n in names)VRAM_GB = min(vram)print(f"\nSo GPU: {N_GPU}")if N_GPU >= 2:    print("  -> moi model mot GPU, song song that")else:    print("  -> 2 tien trinh tren cung 1 GPU (Colab tieu chuan chi cap 1 GPU)")if not IS_A100:    print(f"\n[canh bao] Khong phai A100. Batch size mac dinh duoc tinh cho A100 "          f"40GB; voi {VRAM_GB:.0f} GB cell 9 se tu ha xuong.")

## 2. Cài đặtGhim đúng version đã kiểm chứng. `yolo26` chỉ có từ ultralytics 8.4.x — bản cũhơn sẽ báo "model not found" cho hai model v26.

In [ ]:
%pip install -q "ultralytics==8.4.118" onnx onnxslim onnxruntimeimport ultralytics, torch, platformprint("ultralytics", ultralytics.__version__)print("torch      ", torch.__version__, "| cuda", torch.version.cuda)print("python     ", platform.python_version())from ultralytics.utils.downloads import GITHUB_ASSETS_NAMESfor w in ["yolov8n.pt", "yolo11n.pt", "yolo26n.pt", "yolo26n-p2.pt"]:    print(f"  {w:16s} {'co pretrained' if w in GITHUB_ASSETS_NAMES else 'KHONG co (dung .load())'}")

## 3. Mount Drive**Trước khi chạy cell này**, mở link dataset rồi bấm**Add shortcut to Drive → My Drive**. Thư mục chia sẻ không tự xuất hiện trong`MyDrive` nếu chưa tạo shortcut, và `gdown --folder` bị chặn ở 50 file nênkhông dùng được cho dataset vài nghìn ảnh.

In [ ]:
from google.colab import drivedrive.mount("/content/drive")import osROOT = "/content/drive/MyDrive"print("Thu muc cap 1 trong MyDrive:\n")for d in sorted(os.listdir(ROOT))[:60]:    p = os.path.join(ROOT, d)    if os.path.isdir(p):        print("  ", d)

## 4. Tìm và soi datasetĐiền tên thư mục vào `DATASET_DIR` (lấy từ danh sách cell 3). Cell này **khôngtrain gì cả** — nó chỉ đọc cấu trúc và cho bạn thấy dataset thực sự chứa gì,vì mọi lỗi dataset đều rẻ khi phát hiện ở đây và rất đắt khi phát hiện sau 3tiếng train.

In [ ]:
import os, glob, collections# <<< SUA DONG NAY >>>DATASET_DIR = "/content/drive/MyDrive/TEN_THU_MUC_DATASET"assert os.path.isdir(DATASET_DIR), (    f"Khong thay {DATASET_DIR}. Kiem tra lai ten o cell 3, "    "va da 'Add shortcut to Drive' chua.")IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG")def tree(root, depth=0, maxdepth=2):    if depth > maxdepth:        return    try:        entries = sorted(os.listdir(root))    except PermissionError:        return    for e in entries[:25]:        p = os.path.join(root, e)        if os.path.isdir(p):            n_img = sum(1 for f in os.listdir(p)                        if f.endswith(IMG_EXT)) if depth < maxdepth else 0            n_txt = sum(1 for f in os.listdir(p)                        if f.endswith(".txt")) if depth < maxdepth else 0            tag = []            if n_img:                tag.append(f"{n_img} anh")            if n_txt:                tag.append(f"{n_txt} txt")            print("  " * depth + f"[{e}]" + ("  <- " + ", ".join(tag) if tag else ""))            tree(p, depth + 1, maxdepth)        elif depth == 0:            print("  " * depth + e)print(f"Cau truc {DATASET_DIR}:\n")tree(DATASET_DIR)archives = [p for p in glob.glob(os.path.join(DATASET_DIR, "**", "*"), recursive=True)            if p.endswith((".zip", ".tar", ".tar.gz", ".tgz"))]if archives:    print("\nFile nen tim thay (stage se nhanh hon nhieu):")    for a in archives[:10]:        print(f"  {os.path.basename(a)}  {os.path.getsize(a)/1e9:.2f} GB")yamls = glob.glob(os.path.join(DATASET_DIR, "**", "*.yaml"), recursive=True)if yamls:    print("\nCo san data.yaml:")    for y in yamls[:5]:        print(f"\n--- {y} ---")        print(open(y).read()[:800])

## 5. Chép dataset về đĩa local**Đừng train trực tiếp trên Drive.** Drive gắn qua FUSE; mỗi lần mở một fileảnh là một lượt gọi mạng. Train nano model ở batch 96 cần vài trăm ảnh mỗigiây, và FUSE không đáp ứng nổi — GPU sẽ ngồi chờ, tưởng là model chậm nhưngthật ra là I/O.Chép một lần sang `/content` rồi train từ đó. Có file nén thì chép file nén(một file lớn qua FUSE nhanh hơn hàng nghìn file nhỏ rất nhiều).

In [ ]:
import os, time, shutil, subprocess, globLOCAL = "/content/dataset"os.makedirs(LOCAL, exist_ok=True)# Neu dataset da co san dang nen thi dien duong dan vao day, de trong thi chep ca cay thu muc.ARCHIVE = ""   # vd: "/content/drive/MyDrive/xxx/visdrone.zip"t0 = time.time()if ARCHIVE:    print(f"Chep {os.path.basename(ARCHIVE)} ...")    local_arc = os.path.join("/content", os.path.basename(ARCHIVE))    shutil.copy(ARCHIVE, local_arc)    print(f"  xong sau {time.time()-t0:.0f}s, dang giai nen ...")    shutil.unpack_archive(local_arc, LOCAL)    os.remove(local_arc)else:    print(f"Chep cay thu muc (cham hon, kien nhan) ...")    subprocess.run(["cp", "-r", DATASET_DIR + "/.", LOCAL], check=True)el = time.time() - t0n_img = sum(1 for p in glob.glob(os.path.join(LOCAL, "**", "*"), recursive=True)            if p.endswith(IMG_EXT))sz = subprocess.run(["du", "-sh", LOCAL], capture_output=True, text=True).stdout.split()[0]print(f"\n[ok] {n_img} anh, {sz}, mat {el/60:.1f} phut")print(f"[ok] dataset local: {LOCAL}")

## 6. Dựng và **kiểm tra** `data.yaml`Cell này tự dò cấu trúc, dựng `data.yaml`, rồi chạy bốn kiểm tra mà nếu bỏ quasẽ dẫn tới những kiểu hỏng rất khó truy:| Kiểm tra | Bỏ qua thì sao ||---|---|| Ảnh có label khớp | Ảnh không label bị coi là "không có vật thể" → model học cách bỏ sót || Toạ độ nằm trong `[0,1]` | Label pixel thay vì chuẩn hoá → loss vẫn giảm, mAP vẫn ~0 || `class id < nc` | Index out of range giữa chừng epoch || Histogram lớp | Lớp trống làm AP của lớp đó = −1, kéo lệch mAP |

In [ ]:
import os, glob, yaml, random, collectionsdef find_split(root, split):    """Tim thu muc anh cua mot split, chap nhan nhieu kieu dat ten."""    pats = [f"{split}/images", f"images/{split}", f"{split}",            f"*{split}*/images", f"images/*{split}*"]    for p in pats:        hits = [h for h in glob.glob(os.path.join(root, p)) if os.path.isdir(h)]        for h in hits:            if any(f.endswith(IMG_EXT) for f in os.listdir(h)):                return h    return Nonesplits = {s: find_split(LOCAL, s) for s in ("train", "val", "test")}for s, p in splits.items():    n = len([f for f in os.listdir(p) if f.endswith(IMG_EXT)]) if p else 0    print(f"  {s:6s} {p or '(khong tim thay)'}  {n} anh")assert splits["train"], "Khong tim thay split train. Sua tay bien `splits` roi chay lai."if not splits["val"]:    raise SystemExit("Khong co val. Khong co val thi khong co so lieu de bao cao.")def label_dir_for(img_dir):    cand = img_dir.replace("/images", "/labels")    if os.path.isdir(cand):        return cand    cand2 = os.path.join(os.path.dirname(img_dir), "labels")    return cand2 if os.path.isdir(cand2) else None# --- doc ten lop: uu tien data.yaml co san, neu khong thi suy tu label ---names = Nonefor y in glob.glob(os.path.join(LOCAL, "**", "*.yaml"), recursive=True):    d = yaml.safe_load(open(y))    if isinstance(d, dict) and "names" in d:        names = d["names"]        print(f"\n[ok] lay ten lop tu {y}")        breakif names is None:    print("\n[canh bao] khong co data.yaml -> dung ten lop VisDrone mac dinh")    names = ["pedestrian", "people", "bicycle", "car", "van", "truck",             "tricycle", "awning-tricycle", "bus", "motor"]if isinstance(names, dict):    names = [names[k] for k in sorted(names)]NC = len(names)print(f"[ok] nc = {NC}: {names}")# --- bon kiem tra ---problems = []hist = collections.Counter()for split, img_dir in splits.items():    if not img_dir:        continue    lab_dir = label_dir_for(img_dir)    if not lab_dir:        problems.append(f"{split}: khong tim thay thu muc labels")        continue    imgs = [f for f in os.listdir(img_dir) if f.endswith(IMG_EXT)]    missing = 0    for f in imgs:        stem = os.path.splitext(f)[0]        if not os.path.exists(os.path.join(lab_dir, stem + ".txt")):            missing += 1    if missing:        problems.append(f"{split}: {missing}/{len(imgs)} anh khong co file label")    bad_range, bad_cls, n_box = 0, 0, 0    for lf in random.Random(0).sample(            glob.glob(os.path.join(lab_dir, "*.txt")),            min(400, len(glob.glob(os.path.join(lab_dir, "*.txt"))))):        for line in open(lf):            parts = line.split()            if len(parts) < 5:                continue            n_box += 1            c = int(float(parts[0]))            hist[c] += 1            if c >= NC or c < 0:                bad_cls += 1            if any(not (0.0 <= float(v) <= 1.0) for v in parts[1:5]):                bad_range += 1    if bad_range:        problems.append(f"{split}: {bad_range}/{n_box} box co toa do ngoai [0,1] "                        f"-> label chua chuan hoa")    if bad_cls:        problems.append(f"{split}: {bad_cls}/{n_box} box co class id >= nc={NC}")print("\nPhan bo lop (mau 400 file/split):")for c in range(NC):    bar = "#" * int(40 * hist[c] / max(hist.values())) if hist else ""    print(f"  {c:2d} {names[c]:18s} {hist[c]:7d} {bar}")    if hist[c] == 0:        problems.append(f"lop {c} ({names[c]}) khong co box nao -> AP lop nay se la -1")DATA_YAML = "/content/data.yaml"cfg = {"path": LOCAL, "train": splits["train"], "val": splits["val"], "nc": NC,       "names": names}if splits["test"]:    cfg["test"] = splits["test"]yaml.safe_dump(cfg, open(DATA_YAML, "w"), sort_keys=False, allow_unicode=True)print("\n" + "=" * 60)if problems:    print("VAN DE PHAT HIEN DUOC - doc ky truoc khi train:")    for p in problems:        print("  !!", p)else:    print("Khong phat hien van de nao.")print("=" * 60)print(f"\n[ok] {DATA_YAML}")print(open(DATA_YAML).read())

## 7. Cấu hình`epochs` của **v26n-p2 cao hơn** ba model kia — đây là chủ ý, không phải nhầm.Nó khởi tạo với chỉ ~50% trọng số pretrained (xem đầu notebook), nên cho cùngsố epoch là so sánh không công bằng với chính nó.Ghép cặp cũng có chủ ý: một job nặng đi với một job nhẹ, để hai tiến trìnhkhông cùng lúc đòi đỉnh bộ nhớ.

In [ ]:
SEED = 0IMGSZ = 640EPOCHS_BASE = 100PATIENCE = 30PROJECT = "/content/runs"MODELS = {    "v8n-base": dict(        cfg="yolov8n.yaml", weights="yolov8n.pt", batch=96,        epochs=EPOCHS_BASE, e2e=False),    "v11n-base": dict(        cfg="yolo11n.yaml", weights="yolo11n.pt", batch=96,        epochs=EPOCHS_BASE, e2e=False),    "v26n-base": dict(        cfg="yolo26n.yaml", weights="yolo26n.pt", batch=96,        epochs=EPOCHS_BASE, e2e=True),    "v26n-p2": dict(        # Khong co yolo26n-p2.pt -> dung tu yaml, nap mot phan tu yolo26n.pt.        # P2 them tang stride-4 (160x160 o 640px) nen ton bo nho hon han -> batch thap hon.        cfg="yolo26n-p2.yaml", weights="yolo26n.pt", batch=48,        epochs=int(EPOCHS_BASE * 1.5), e2e=True),}# Nang tran detection len 500 cho model end-to-end: giao thuc VisDrone cham o# maxDets=500, ma dau end2end mac dinh cat cung o 300. Anh VisDrone dong co the# vuot 300 vat the -> khong sua thi v26 bi thiet ma khong bao loi gi.MAX_DET = 500PAIRS = [("v26n-p2", "v8n-base"), ("v26n-base", "v11n-base")]import torchDEVICES = ([0, 1] if torch.cuda.device_count() >= 2 else [0, 0])print(f"So GPU: {torch.cuda.device_count()}  -> gan device {DEVICES}")for a, b in PAIRS:    print(f"  luot: {a} (GPU {DEVICES[0]}) || {b} (GPU {DEVICES[1]})")

## 8. Preflight — dựng cả 4 model trước khi trainBa tiếng train rồi mới biết một model không dựng được là ba tiếng mất trắng.Cell này dựng cả bốn, in số tham số, **shape output thật**, và **tỉ lệ trọng sốnạp được** — con số cuối chính là bằng chứng cho caveat của v26n-p2.

In [ ]:
import warnings, io, contextlib, re, torchwarnings.filterwarnings("ignore")from ultralytics import YOLOpreflight = {}for tag, spec in MODELS.items():    buf = io.StringIO()    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):        m = YOLO(spec["cfg"], verbose=False)        if spec["weights"]:            m.load(spec["weights"])    # Dem truc tiep tren state_dict thay vi doc log: ultralytics in dong    # "Transferred x/y" qua LOGGER rieng, redirect_stdout khong bat duoc, va    # mot cot im lang bao "n/a" chinh la cot khong ai kiem tra. Day dung la    # tieu chi ultralytics dung de nap: trung ten VA trung shape.    if spec["weights"]:        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):            src_sd = YOLO(spec["weights"]).model.state_dict()        dst_sd = m.model.state_dict()        matched = sum(1 for k, v in dst_sd.items()                      if k in src_sd and src_sd[k].shape == v.shape)        transferred, frac = f"{matched}/{len(dst_sd)}", matched / len(dst_sd)    else:        transferred, frac = "0/0 (tu dau)", 0.0    head = m.model.model[-1]    if getattr(head, "end2end", False):        head.max_det = MAX_DET    m.model.eval()    with torch.no_grad():        y = m.model(torch.zeros(1, 3, IMGSZ, IMGSZ))    out = y[0] if isinstance(y, (list, tuple)) else y    n_par = sum(p.numel() for p in m.model.parameters())    is_e2e = bool(getattr(head, "end2end", False))    # Preflight dung nc mac dinh cua yaml (80, COCO) vi model chua gap dataset.    # Sau khi train tren nc=NC, shape se doi voi model can NMS. Ghi ca hai de    # khong ai doc nham con so 84 nay thanh shape cuoi cung.    expected = (1, MAX_DET, 6) if is_e2e else (1, 4 + NC, out.shape[-1])    preflight[tag] = dict(params=n_par, out_shape_preflight=tuple(out.shape),                          out_shape=expected,                          transferred=transferred, transfer_frac=frac,                          end2end=is_e2e)    del m    torch.cuda.empty_cache()print(f"{'model':12s} {'params':>9s} {'output sau train':>20s} {'NMS':>6s} {'weights nap':>12s}")print("-" * 66)for t, p in preflight.items():    print(f"{t:12s} {p['params']/1e6:8.2f}M {str(p['out_shape']):>20s} "          f"{'khong' if p['end2end'] else 'can':>6s} {p['transferred']:>12s}")print(f"\n(Cot output la shape DU KIEN voi nc={NC}. Preflight chay o nc=80 mac dinh "      f"cua yaml,\n nen shape thuc do duoc luc nay la "      f"{preflight[list(MODELS)[0]]['out_shape_preflight']} - khong phai con so cuoi.\n"      f" Shape that duoc ghi lai tu file ONNX o cell 15.)")# Nguong 0.75, khong phai 0.9: model base thuong chi dat ~90% vi dau detect# khong khop khi nc khac COCO - do la binh thuong. Chi truong hop mat ca# backbone/neck (p2, ~50%) moi la dieu can canh bao.low = [t for t, p in preflight.items() if p["transfer_frac"] < 0.75]if low:    print(f"\n[luu y] {', '.join(low)}: mot phan lon model khoi tao ngau nhien. "          f"Da bu bang epochs cao hon o cell 7, nhung bang so sanh cuoi VAN phai "          f"ghi ro - neu khong, doc bang se ra ket luan sai ve kien truc.")

## 9. Đo bộ nhớ thật, không đoánBatch size ở cell 7 là ước lượng cho A100 40 GB. Cell này **đo thật**: chạy vàibước forward+backward với dữ liệu ngẫu nhiên rồi đọc đỉnh bộ nhớ, và tự hạbatch nếu cặp ghép không vừa.Đây là con số ước lượng cận dưới (chưa tính EMA và optimizer state, vốn nhỏ vớimodel nano), nhưng nó bắt được lỗi OOM ở phút thứ hai thay vì giờ thứ hai.

In [ ]:
import torch, gc, warningswarnings.filterwarnings("ignore")from ultralytics import YOLOdef _all_tensors(o):    """Dau train tra ve tensor, list, hoac dict (yolo26 tra one2many/one2one).    Gom het lai roi tinh mot loss gia - chi can do bo nho, khong can dung."""    if torch.is_tensor(o):        return [o]    if isinstance(o, dict):        o = list(o.values())    if isinstance(o, (list, tuple)):        out = []        for x in o:            out.extend(_all_tensors(x))        return out    return []def peak_mem_gb(cfg, batch, imgsz=IMGSZ, steps=3):    torch.cuda.empty_cache(); gc.collect()    torch.cuda.reset_peak_memory_stats()    m = YOLO(cfg, verbose=False).model.cuda().train()    opt = torch.optim.SGD(m.parameters(), lr=1e-4)    scaler = torch.amp.GradScaler("cuda")    try:        for _ in range(steps):            x = torch.rand(batch, 3, imgsz, imgsz, device="cuda")            with torch.amp.autocast("cuda"):                ts = [t for t in _all_tensors(m(x)) if t.is_floating_point()]                if not ts:                    raise RuntimeError(f"{cfg}: khong lay duoc tensor nao tu dau ra")                loss = sum((t.float() ** 2).mean() for t in ts)            scaler.scale(loss).backward()            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)        peak = torch.cuda.max_memory_allocated() / 1e9    except torch.cuda.OutOfMemoryError:        peak = float("inf")    finally:        del m, opt        torch.cuda.empty_cache(); gc.collect()    return peakTOTAL_GB = torch.cuda.get_device_properties(0).total_memory / 1e9BUDGET = TOTAL_GB * 0.85          # chua 15% cho fragmentation va cudnn workspaceprint(f"VRAM {TOTAL_GB:.1f} GB, ngan sach dung {BUDGET:.1f} GB\n")measured = {}for tag, spec in MODELS.items():    p = peak_mem_gb(spec["cfg"], spec["batch"])    measured[tag] = p    print(f"  {tag:12s} batch {spec['batch']:3d} -> dinh {p:5.1f} GB")print()same_gpu = (DEVICES[0] == DEVICES[1])for a, b in PAIRS:    need = measured[a] + measured[b] if same_gpu else max(measured[a], measured[b])    ok = need <= BUDGET    print(f"  cap ({a}, {b}): can {need:.1f} GB {'<=' if ok else '>'} {BUDGET:.1f} GB "          f"{'OK' if ok else '-> ha batch'}")    while not ok and min(MODELS[a]["batch"], MODELS[b]["batch"]) > 8:        heavy = a if measured[a] >= measured[b] else b        MODELS[heavy]["batch"] = max(8, MODELS[heavy]["batch"] // 2)        measured[heavy] = peak_mem_gb(MODELS[heavy]["cfg"], MODELS[heavy]["batch"])        need = measured[a] + measured[b] if same_gpu else max(measured[a], measured[b])        ok = need <= BUDGET        print(f"     ha {heavy} xuong batch {MODELS[heavy]['batch']} -> {need:.1f} GB")print("\nBatch chot lai:")for t, s in MODELS.items():    print(f"  {t:12s} batch {s['batch']}")

## 10. Script trainViết ra file để chạy được như tiến trình độc lập — đó là cách duy nhất chạysong song hai model thật sự, và cũng khiến một job chết không kéo theo job kia.Có `resume`: Colab ngắt thì chạy lại đúng cell đó, nó đọc `last.pt` và đi tiếp.

In [ ]:
%%writefile /content/train_worker.pyimport argparse, json, os, sys, warningswarnings.filterwarnings("ignore")ap = argparse.ArgumentParser()ap.add_argument("--tag", required=True)ap.add_argument("--cfg", required=True)ap.add_argument("--weights", default="")ap.add_argument("--data", required=True)ap.add_argument("--epochs", type=int, required=True)ap.add_argument("--batch", type=int, required=True)ap.add_argument("--imgsz", type=int, default=640)ap.add_argument("--device", default="0")ap.add_argument("--project", required=True)ap.add_argument("--seed", type=int, default=0)ap.add_argument("--patience", type=int, default=30)ap.add_argument("--workers", type=int, default=6)ap.add_argument("--max-det", type=int, default=500)a = ap.parse_args()os.environ["CUDA_VISIBLE_DEVICES"] = a.deviceos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")import torchfrom ultralytics import YOLOlast = os.path.join(a.project, a.tag, "weights", "last.pt")if os.path.exists(last):    print(f"[{a.tag}] tim thay {last} -> resume", flush=True)    model = YOLO(last)    resume = Trueelse:    model = YOLO(a.cfg, verbose=False)    if a.weights:        model.load(a.weights)    resume = Falsehead = model.model.model[-1]if getattr(head, "end2end", False):    head.max_det = a.max_det    print(f"[{a.tag}] dau end2end: max_det 300 -> {a.max_det}", flush=True)print(f"[{a.tag}] bat dau: batch={a.batch} epochs={a.epochs} "      f"device=cuda:0 (that: GPU {a.device})", flush=True)model.train(    data=a.data, epochs=a.epochs, imgsz=a.imgsz, batch=a.batch,    device=0, workers=a.workers, seed=a.seed, deterministic=False,    project=a.project, name=a.tag, exist_ok=True, resume=resume,    patience=a.patience, amp=True, cache=False, val=True, plots=True,    # Augment: giu photometric, han che geometric manh vi vat the VisDrone rat nho    # -> scale/shear lon lam mat hoan toan cac box duoi 10px.    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,    degrees=0.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0,    flipud=0.0, fliplr=0.5, mosaic=1.0, mixup=0.0, close_mosaic=10,)res = model.val(data=a.data, imgsz=a.imgsz, batch=a.batch, device=0,                max_det=a.max_det, verbose=False)summary = {    "tag": a.tag, "mAP50-95": float(res.box.map), "mAP50": float(res.box.map50),    "mAP75": float(res.box.map75),    "maps_per_class": [float(x) for x in res.box.maps],    "epochs": a.epochs, "batch": a.batch, "imgsz": a.imgsz, "seed": a.seed,    "max_det": a.max_det,    "best": os.path.join(a.project, a.tag, "weights", "best.pt"),}with open(os.path.join(a.project, a.tag, "summary.json"), "w") as f:    json.dump(summary, f, indent=2)print(f"[{a.tag}] XONG mAP50-95={res.box.map:.4f} mAP50={res.box.map50:.4f}", flush=True)

## 11. Hàm chạy song songĐọc log của cả hai job trong lúc chúng chạy, và **không giấu lỗi**: job nàochết thì in ra 40 dòng cuối của nó ngay.

In [ ]:
import subprocess, sys, time, os, itertoolsdef launch_pair(pair, devices):    procs, logs = {}, {}    for tag, dev in zip(pair, devices):        spec = MODELS[tag]        os.makedirs(os.path.join(PROJECT, tag), exist_ok=True)        logp = f"/content/log_{tag}.txt"        logs[tag] = logp        cmd = [            sys.executable, "/content/train_worker.py",            "--tag", tag, "--cfg", spec["cfg"], "--weights", spec["weights"],            "--data", DATA_YAML, "--epochs", str(spec["epochs"]),            "--batch", str(spec["batch"]), "--imgsz", str(IMGSZ),            "--device", str(dev), "--project", PROJECT, "--seed", str(SEED),            "--patience", str(PATIENCE), "--max-det", str(MAX_DET),            "--workers", "6" if devices[0] != devices[1] else "4",        ]        f = open(logp, "w")        procs[tag] = (subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT), f)        print(f"[launch] {tag} -> GPU {dev}, log {logp}")        time.sleep(20)   # lech nhau de hai job khong cung luc cap phat bo nho    spin = itertools.cycle("|/-\\")    t0 = time.time()    while any(p.poll() is None for p, _ in procs.values()):        time.sleep(30)        state = []        for tag, (p, _) in procs.items():            tail = ""            try:                lines = [l for l in open(logs[tag]).read().splitlines() if l.strip()]                for l in reversed(lines):                    if "/" in l and ("it/s" in l or "s/it" in l or "G " in l):                        tail = l.strip()[:46]                        break            except Exception:                pass            state.append(f"{tag}: {'chay' if p.poll() is None else 'xong'} {tail}")        print(f"\r{next(spin)} {(time.time()-t0)/60:6.1f} phut | " +              " || ".join(state)[:150], end="", flush=True)    print()    for tag, (p, f) in procs.items():        f.close()        if p.returncode != 0:            print(f"\n!!! {tag} loi (exit {p.returncode}). 40 dong cuoi:\n")            print("\n".join(open(logs[tag]).read().splitlines()[-40:]))        else:            print(f"[ok] {tag} hoan tat")    return procsprint("San sang. Chay cell 12 de bat dau luot 1.")

## 12. Lượt 1 — `v26n-p2` + `v8n-base`Vài giờ. Colab ngắt thì chạy lại chính cell này, nó tự resume.

In [ ]:
_ = launch_pair(PAIRS[0], DEVICES)

## 13. Lượt 2 — `v26n-base` + `v11n-base`

In [ ]:
_ = launch_pair(PAIRS[1], DEVICES)

## 14. Bảng so sánhĐọc `summary.json` của từng job. Bảng có cột **`weights nạp`** để người đọcthấy ngay v26n-p2 không xuất phát cùng vạch — nếu không có cột đó, bảng này sẽbị đọc sai.

In [ ]:
import json, osimport pandas as pdrows = []for tag in MODELS:    sp = os.path.join(PROJECT, tag, "summary.json")    if not os.path.exists(sp):        print(f"[thieu] {tag} chua co summary.json")        continue    s = json.load(open(sp))    pf = preflight[tag]    rows.append({        "model": tag,        "mAP50-95": round(s["mAP50-95"], 4),        "mAP50": round(s["mAP50"], 4),        "mAP75": round(s["mAP75"], 4),        "params(M)": round(pf["params"] / 1e6, 2),        "epochs": s["epochs"],        "batch": s["batch"],        "max_det": s["max_det"],        "NMS": "khong" if pf["end2end"] else "can",        "weights nap": pf["transferred"],        "output": str(pf["out_shape"]),    })df = pd.DataFrame(rows).sort_values("mAP50-95", ascending=False)display(df)df.to_csv("/content/comparison.csv", index=False)print("\nDoc bang nay can nho:")print("  - v26n-p2 chi nap duoc ~50% pretrained -> da bu bang epochs cao hon,")print("    nhung van khong phai so sanh hoan toan cong bang.")print("  - Cot 'output' khac nhau giua v8/v11 va v26: pipeline hien tai chi")print("    decode duoc dang (1, 4+nc, A). Xem cell 15.")

## 15. Export ONNX cho pipeline QCS8550Dùng đúng thiết lập mà pipeline trên board yêu cầu, và vá sẵn lỗi đã gặp:- `opset=13` — opset mới sinh op mà QNN đẩy ngược về CPU- `dynamic=False`, shape tĩnh — bắt buộc để tạo QNN context binary- `simplify=True`- **`sanitise_onnx()`** — Ultralytics + onnxslim để tensor output nằm cả trong  `graph.output` lẫn `value_info`. ONNX Runtime bỏ qua, còn AI Hub **từ chối  compile**: `Tensors {'output0'} occur in value_info but also in model IO`.  Đây là lỗi thật đã chặn pipeline lần trước.- `nms=False` cho v8/v11 (NMS chạy trên Kryo và được đo riêng). v26 không có  NMS để mà tắt.

In [ ]:
import os, json, hashlib, shutil, warningswarnings.filterwarnings("ignore")from ultralytics import YOLOdef sha256(path, chunk=1 << 20):    h = hashlib.sha256()    with open(path, "rb") as f:        for b in iter(lambda: f.read(chunk), b""):            h.update(b)    return h.hexdigest()def sanitise_onnx(path):    """Bo tensor vua nam trong graph IO vua nam trong value_info (AI Hub tu choi)."""    import onnx    m = onnx.load(path)    io_names = {t.name for t in m.graph.input} | {t.name for t in m.graph.output}    dupes = [vi.name for vi in m.graph.value_info if vi.name in io_names]    if dupes:        keep = [vi for vi in m.graph.value_info if vi.name not in io_names]        del m.graph.value_info[:]        m.graph.value_info.extend(keep)        onnx.save(m, path)    return dupesOUT = "/content/export"os.makedirs(OUT, exist_ok=True)manifest = []for tag, spec in MODELS.items():    best = os.path.join(PROJECT, tag, "weights", "best.pt")    if not os.path.exists(best):        print(f"[bo qua] {tag}: chua co best.pt")        continue    m = YOLO(best)    head = m.model.model[-1]    is_e2e = bool(getattr(head, "end2end", False))    if is_e2e:        head.max_det = MAX_DET    kw = dict(format="onnx", imgsz=IMGSZ, opset=13, dynamic=False,              simplify=True, batch=1)    if not is_e2e:        # Chi co y nghia voi dau can NMS. Truyen nms=False cho model end2end        # la vo nghia (khong co NMS de tat) va co the lam export bao loi.        kw["nms"] = False    p = m.export(**kw)    onnx_path = os.path.join(OUT, f"{tag}.onnx")    shutil.move(str(p), onnx_path)    dupes = sanitise_onnx(onnx_path)    import onnxruntime as ort    sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])    ishape = sess.get_inputs()[0].shape    oshape = sess.get_outputs()[0].shape    pt_path = os.path.join(OUT, f"{tag}.pt")    shutil.copy(best, pt_path)    rec = {        "tag": tag, "onnx": os.path.basename(onnx_path),        "pt": os.path.basename(pt_path),        "onnx_sha256_16": sha256(onnx_path)[:16],        "pt_sha256_16": sha256(pt_path)[:16],        "input_shape": ishape, "output_shape": oshape,        "end2end_no_nms": bool(preflight[tag]["end2end"]),        "max_det": MAX_DET if preflight[tag]["end2end"] else None,        "opset": 13, "imgsz": IMGSZ,        "value_info_dupes_removed": len(dupes),        "pipeline_compatible": (not preflight[tag]["end2end"]),    }    manifest.append(rec)    print(f"[ok] {tag:12s} output {oshape}  sha {rec['onnx_sha256_16']}  "          f"{'(da vá ' + str(len(dupes)) + ' dupe)' if dupes else ''}")json.dump(manifest, open(os.path.join(OUT, "manifest.json"), "w"), indent=2)print("\n" + "=" * 68)print("TUONG THICH VOI 3-pipeline/detector.py")print("=" * 68)for r in manifest:    if r["pipeline_compatible"]:        print(f"  {r['tag']:12s} OK   {r['output_shape']} - decode nhu cu")    else:        print(f"  {r['tag']:12s} KHAC {r['output_shape']} - box da decode san, "              f"KHONG chay NMS nua")print("\nCam vao pipeline ma khong sua postprocess thi v26 se sai tham lang,")print("khong bao loi. Can them mot nhanh decode cho dang (1, N, 6).")

## 16. Lưu về Drive

In [ ]:
import os, shutil, json, datetimeSAVE_DIR = "/content/drive/MyDrive/skysentry/finetune_4models"os.makedirs(SAVE_DIR, exist_ok=True)for f in os.listdir(OUT):    shutil.copy(os.path.join(OUT, f), os.path.join(SAVE_DIR, f))for tag in MODELS:    d = os.path.join(PROJECT, tag)    if not os.path.isdir(d):        continue    dst = os.path.join(SAVE_DIR, "runs", tag)    os.makedirs(dst, exist_ok=True)    for f in ("results.csv", "args.yaml", "summary.json"):        s = os.path.join(d, f)        if os.path.exists(s):            shutil.copy(s, dst)if os.path.exists("/content/comparison.csv"):    shutil.copy("/content/comparison.csv", SAVE_DIR)shutil.copy(DATA_YAML, SAVE_DIR)run_meta = {    "ngay": datetime.datetime.now().isoformat(timespec="seconds"),    "dataset": DATASET_DIR,    "nc": NC, "names": names, "imgsz": IMGSZ, "seed": SEED,    "max_det": MAX_DET,    "gpu": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],    "ultralytics": ultralytics.__version__,    "models": {t: {k: v for k, v in s.items()} for t, s in MODELS.items()},    "preflight": {t: {k: (list(v) if isinstance(v, tuple) else v)                      for k, v in p.items()} for t, p in preflight.items()},}json.dump(run_meta, open(os.path.join(SAVE_DIR, "run_meta.json"), "w"),          indent=2, ensure_ascii=False)print(f"[ok] da luu vao {SAVE_DIR}\n")for f in sorted(os.listdir(SAVE_DIR)):    p = os.path.join(SAVE_DIR, f)    if os.path.isfile(p):        print(f"  {f:28s} {os.path.getsize(p)/1e6:7.2f} MB")    else:        print(f"  {f}/")

---## Xong rồi thì làm gì tiếp**Việc bắt buộc trước khi dùng v26 trong pipeline.** `3-pipeline/detector.py`hiện chỉ decode `(1, 4+nc, A)`. Hai model v26 trả `(1, N, 6)` đã decode sẵn.Cắm vào mà không sửa thì **không có lỗi nào được báo** — chỉ là kết quả sai.Cần thêm một nhánh: nếu `raw.shape[-1] == 6` thì tách`[x1, y1, x2, y2, conf, cls]`, bỏ NMS, chỉ lọc theo `conf` rồi đưa toạ độ về hệảnh gốc bằng `gain`/`pad` như cũ.**Việc đáng làm nhất sau đó.** Bỏ được NMS là bỏ được **18 ms/frame** trên CPU— so với 47 ms inference thì đó là khoản cắt lớn nhất còn lại trong framebudget. Nhưng phải kiểm tra thật: đầu end-to-end dùng topk, và topk có thể bịQNN đẩy về CPU. Nếu `n_ops_fallback > 0` thì phần tiết kiệm được sẽ bị trả lạiở chỗ khác. Chạy compile job trên AI Hub rồi đọc `n_ops_fallback` trước khitin vào con số này.**Các con số cần ghi kèm mỗi dòng kết quả**, nếu không thì hai bảng không sođược với nhau: `max_det` (500, không phải 300 mặc định), tỉ lệ trọng số nạpđược, `imgsz`, `seed`, và sha256 của ONNX.